# Islanding the Grid (QAOA)

*QAOA primer · UTC Problem 2 · recommended before the TVA tracks*

Split a grid into two islands so you cut the most connections between them (Max-Cut). Solve it with QAOA, then scale up on qBraid's GPU simulator.

**Your job:** fill in the cost layer and the optimizer loop, solve the 6-node grid, then run the larger graph on GPU and compare against the classical baseline.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

sim = AerSimulator()


In [ ]:
# The grids (nodes are substations, edges are power lines)

# 6-node grid (2 x 3) for Part 1
edges_small = [(0,1),(1,2),(3,4),(4,5),(0,3),(1,4),(2,5)]
n_small = 6

# 12-node grid (3 x 4) for the GPU scale-up in Part 2
edges_large = [(0,1),(1,2),(2,3),(4,5),(5,6),(6,7),(8,9),(9,10),(10,11),
               (0,4),(4,8),(1,5),(5,9),(2,6),(6,10),(3,7),(7,11)]
n_large = 12


In [ ]:
def cut_value(bitstring, edges):
    """Count how many edges are cut by a 0/1 assignment (given as a string)."""
    bits = [int(b) for b in bitstring]
    return sum(1 for (u, v) in edges if bits[u] != bits[v])


In [ ]:
def qaoa_circuit(params, n, edges, p):
    """Build a p-layer QAOA circuit. params = [gammas..., betas...]."""
    gammas = params[:p]
    betas = params[p:]
    qc = QuantumCircuit(n, n)
    qc.h(range(n))                    # start in equal superposition
    for layer in range(p):
        # --- Cost layer ---
        # TODO: for each edge (u, v), apply the Max-Cut cost rotation.
        # Hint: cx(u, v); rz(2 * gammas[layer], v); cx(u, v)
        raise NotImplementedError("Implement the cost layer")

        # --- Mixer layer (provided) ---
        for q in range(n):
            qc.rx(2 * betas[layer], q)
    qc.measure(range(n), range(n))
    return qc


In [ ]:
def expected_cut(params, n, edges, p, shots=512):
    """Run the circuit and return the AVERAGE cut value (we maximize this)."""
    qc = qaoa_circuit(params, n, edges, p)
    counts = sim.run(transpile(qc, sim), shots=shots).result().get_counts()
    total = sum(counts.values())
    # bitstrings come back reversed in Qiskit; reverse to match node order
    avg = sum(cut_value(bs[::-1], edges) * c for bs, c in counts.items()) / total
    return avg


In [ ]:
# Optimize (we minimize the NEGATIVE average cut)
p = 1                                  # number of QAOA layers; try 1, then 2
n, edges = n_small, edges_small

def objective(params):
    return -expected_cut(params, n, edges, p)

# TODO: choose a starting point and call scipy.optimize.minimize.
# Hint: x0 = np.random.rand(2 * p); result = minimize(objective, x0, method="COBYLA")
raise NotImplementedError("Run the optimizer, then read out the best partition below")


In [ ]:
# Read out the best partition (run after the optimizer works)
best_params = result.x
qc = qaoa_circuit(best_params, n, edges, p)
counts = sim.run(transpile(qc, sim), shots=2048).result().get_counts()
best_bits = max(counts, key=counts.get)[::-1]
island_A = [i for i, b in enumerate(best_bits) if b == '0']
island_B = [i for i, b in enumerate(best_bits) if b == '1']
print("Island A:", island_A)
print("Island B:", island_B)
print("Cut value:", cut_value(best_bits, edges))


In [ ]:
# Classical baseline (brute force) — provided, for the small graph only
from itertools import product
def brute_force(n, edges):
    best_cut, best = -1, None
    for assign in product([0,1], repeat=n):
        c = cut_value("".join(map(str, assign)), edges)
        if c > best_cut:
            best_cut, best = c, assign
    return best_cut, best
print("Best possible cut (classical):", brute_force(n_small, edges_small)[0])


### GPU scale-up (Part 2)

Switch to the 12-node grid and re-run the optimizer cells. The next cell uses a GPU if your qBraid Lab instance has one with the GPU build of Qiskit Aer; otherwise it stays on the CPU, where 12 qubits still runs in seconds. Note how the runtime changes either way.

In [ ]:
import time

if "GPU" in AerSimulator().available_devices():
    sim = AerSimulator(device="GPU")
    print("Using the GPU simulator")
else:
    sim = AerSimulator()
    print("No GPU on this instance, so staying on the CPU simulator")

n, edges = n_large, edges_large
start = time.time()
# ... now re-run your optimizer cell (with n, edges set above), then:
# print(f"12-node run took {time.time() - start:.1f} s")

## What to submit
- **Part 1:** the 6-node partition and cut value.
- **Part 2:** the 12-node result run on GPU, with a runtime note.
- **Part 3:** the classical baseline comparison.
- **Part 4:** the written reality-check, in your write-up.
